<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="30%"></a>
</p>


<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>

# Docker Contexts and Local Targets: choosing the right host for a command

When several Docker hosts are available, the same command can query different images and containers depending on its target. You run `docker container ls`, but the container you expected is missing. Was it removed, was it never there, or is something else going on?

A __Docker context__ is a named connection configuration for the Docker client. It identifies an endpoint through which the client reaches a daemon. Switching contexts changes where commands go; it does not move images or containers between hosts. Naming that target helps you find the resources you built and keep practice changes on the base station. 

For example, the base station and a physical Duckiedrone each have their own images and containers. 

In this notebook, we will inspect the local connection, select a context for one command, and create an SSH context to inspect a physical Duckiedrone.

## Inspect the current base-station target

A Docker context tells the Docker client which daemon receives a command. [Notebook 2](./2-docker-containers-images-and-safe-local-practice.ipynb) introduces this target check for the local exercises. Use the local context named by your base-station setup; on many Linux systems it is named `default`. Start by checking the selected context:

```bash
docker context show
```

List the available contexts and their configured endpoints to identify which host each name reaches:

```bash
docker context ls
```

An illustrative listing is:

```shell
$ docker context ls
NAME                 DESCRIPTION                              DOCKER ENDPOINT               ERROR
default *            Current DOCKER_HOST based configuration  unix:///var/run/docker.sock
duckiedrone-mydrone   Duckiedrone connection                   ssh://duckie@mydrone.local
```

Look at the `DOCKER ENDPOINT` column. For example:

- `unix:///var/run/docker.sock` identifies a Unix socket available to the client.

- `ssh://duckie@mydrone.local` identifies a connection through SSH to another computer.

A context's `NAME` is just a label. Inspect its endpoint and relate it to your setup before deciding which daemon it reaches. A socket available inside a development container, for example, may connect to a daemon outside that container. See [Docker contexts](https://docs.docker.com/engine/manage-resources/contexts/).

Remember that the environment variables `DOCKER_HOST` and `DOCKER_CONTEXT` can override the selected context. Inspect their values to understand how an unqualified command will choose its target:

```bash
printf 'DOCKER_HOST=%s\n' "${DOCKER_HOST:-<unset>}"
printf 'DOCKER_CONTEXT=%s\n' "${DOCKER_CONTEXT:-<unset>}"
```

Each line shows `<unset>` for its variable if no override is set. If either variable is set, it overrides the default context selection. `DOCKER_CONTEXT` takes precedence over `DOCKER_HOST`. 

## Make the base-station target explicit

The `--context` option selects a context by name for one Docker command without changing the saved default. For a stored, non-default context, this explicit selection takes precedence over `DOCKER_CONTEXT` and `DOCKER_HOST`. In contrast, `docker context use NAME` changes the saved default for later commands, which environment variables can override.

The built-in `default` context is different: its endpoint can still come from `DOCKER_HOST`, even when you use `--context default`. Selecting that context name does not by itself guarantee a local target.

Assign `LOCAL_CONTEXT` to your base station's local context; `default` below is an example:

```bash
LOCAL_CONTEXT=default
```

Inspect the context's endpoint before contacting its daemon:

```bash
docker context inspect "$LOCAL_CONTEXT"
```

Compare the Docker endpoint in the output with the local practice endpoint verified in your base-station setup. If you selected `default`, also check the `DOCKER_HOST` value inspected earlier. If the endpoint is unfamiliar or is not your expected local practice target, stop and follow your base-station setup instructions rather than querying that daemon.

After confirming the local endpoint, report the client and server versions:

```bash
docker --context "$LOCAL_CONTEXT" version
```

A server section confirms that the daemon at the verified endpoint responded. Compare the next output with the first `context show` result to check that the per-command selection left the default unchanged:

```bash
docker context show
```

Together, the endpoint, override environment values, and version result explain where the client sends commands and whether that daemon is reachable. Use the base-station target for commands that create or remove `lx-docker-*` resources, since those operations act on the selected host's inventory.

## Remote Duckiedrone Docker Contexts

To inspect the Duckiedrone's resources from the base station, you need a connection to its daemon. We will create that connection, compare it with a shell on the Duckiedrone, and inspect both hosts' images and containers to see where each resource lives.

### Create a Duckiedrone context

This activity needs a physical Duckiedrone with configured SSH access. Use a Duckiedrone you own or have permission to access. Confirm its SSH connection using [Linux and Networking on the Duckiedrone](https://github.com/duckietown/lx-dd-linux-and-networking). The remote `duckie` account also needs permission to access the Docker socket: SSH runs the Docker connection command on the Duckiedrone.

Run the context commands in a base-station terminal. If `.local` resolution is unavailable, use the Duckiedrone's IP (Internet Protocol) address in the SSH endpoint. A Docker context stores the remote connection details on the base station; the first command that uses it contacts the Duckiedrone. Create the context:

```bash
docker context create duckiedrone-DUCKIEDRONE_NAME \
  --description "Duckiedrone connection" \
  --docker "host=ssh://duckie@DUCKIEDRONE_NAME.local"
```

Inspect the stored configuration to confirm the SSH hostname before connecting:

```bash
docker context inspect duckiedrone-DUCKIEDRONE_NAME
```

Configure SSH key authentication before running `version`, so the connection does not require an interactive password prompt. Follow the instructions in [Linux and Networking on the Duckiedrone](https://github.com/duckietown/lx-dd-linux-and-networking/blob/ente/notebooks/13-physical-duckiedrone-ssh-access.ipynb).

A server section confirms that the client reached a Docker daemon through the configured SSH endpoint:

```bash
docker --context duckiedrone-DUCKIEDRONE_NAME version
```

The `--context` option applies to that command. Compare the next output with your earlier local selection to check that the default context stayed unchanged:

```bash
docker context show
```

If the connection fails or times out, check the Duckiedrone's name, network reachability, SSH access, and the `duckie` account's Docker permissions against its setup instructions.

Duckietown's `dts stack` commands select a host and a stack using their own connection settings. Use those commands to start Duckiedrone services as described in [Notebook 8](./8-duckiedrone-docker-hosts-and-stacks.ipynb); the Docker context here supports direct inspection.

### Run the same command two ways

An SSH shell and a base-station context can query the same physical Duckiedrone daemon. Comparing them shows where the Docker client runs: SSH gives you a shell on the Duckiedrone, while `--context` keeps the client on the base station and sends one command to the Duckiedrone.

First, use SSH to open a shell on the Duckiedrone:

```bash
ssh duckie@DUCKIEDRONE_NAME.local
```

Inside the SSH shell, verify that Docker targets the Duckiedrone's local daemon, as you did for the base station. Then inspect the images and containers and exit:

```bash
docker image ls
docker container ls --all
exit
```

Alternatively, remain on the base station and run the same commands through the context:

```bash
docker --context duckiedrone-DUCKIEDRONE_NAME image ls
docker --context duckiedrone-DUCKIEDRONE_NAME container ls --all
```

Both paths query the same Duckiedrone's images and containers when the context points to that Duckiedrone. Timing or client formatting may differ, but the resources belong to the same daemon. The explicit context is useful for a one-command status check or a script that names its target; the SSH shell is useful when you also need other tools on the Duckiedrone.

#### Remove the local context

After completing the comparisons, keep the base station's local context selected and remove the Duckiedrone context when you have finished using it:

```bash
docker context rm duckiedrone-DUCKIEDRONE_NAME
```

Removing the context deletes its saved connection settings on the base station. Containers and images on the Duckiedrone remain available.

### Docker contexts in an editor

An IDE (integrated development environment) such as VS Code (Visual Studio Code) can use Docker contexts to inspect a remote daemon's images and containers. Its container tools can also open a shell inside a running container. That shell belongs to the container; opening a shell on the Duckiedrone host itself requires a separate connection, such as SSH.

For example, if you are using a Duckietown Workspace, you can use the "Containers" extension in VS Code to connect to the Duckiedrone context. The extension displays the selected daemon's containers, images, and volumes. Keep this activity read-only, and use the documented Duckietown workflows for changes to Duckiedrone services.

[Figure 1](#figure-1) shows a selected Duckiedrone context in the Containers extension.

<figure id="figure-1" class="lx-figure">
  <img src="../assets/images/VScode-contexts.jpg" alt="VS Code Containers extension with a Duckiedrone Docker context selected" width="100%" style="max-width:577px; height:auto;">
  <figcaption>Figure 1: The Containers extension displays the selected Duckiedrone context and its Docker resources.</figcaption>
</figure>

## Further reading

See Docker's official guides to [contexts](https://docs.docker.com/engine/manage-resources/contexts/) and [SSH daemon connections](https://docs.docker.com/engine/security/protect-access/), and the [Duckiedrone DD24 manual](https://docs.duckietown.com/ente/opmanual-dd24/) for physical Duckiedrone management.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()


## Summary and conclusions

A context names a daemon connection; switching contexts does not move images or containers. You inspected connection settings and compared a remote context with an SSH shell. Using `--context` selects a target for one command. Next, [see how the Duckietown Shell wraps Docker operations](./11-other-docker-uses-in-duckietown.ipynb) in project workflows.
